# Занятие 4. Лабораторная: метрики, валидация и разведочный анализ данных

Лабораторная домашняя. В ней двадцать задач: четырнадцать обязательных
и шесть бонусных, у бонусных это написано в заголовке. Обязательные задачи
дают до 10 баллов, каждая решенная бонусная — балл сверху, итог не больше 11.
Задачи сгруппированы в семь блоков: первый взгляд на данные, связь признаков
с ответом, кодирование категорий, метрики, валидация, сохранение модели,
тексты.

Почти все, что здесь есть, понадобится в первой домашней работе: таблица
метрик по группам, базовые модели для сравнения, честное разбиение при
повторяющихся объектах, модель, которую можно сохранить и запустить
в другом месте, нормализация русского и английского текста.

**Заготовки.** В каждой заготовке указано, что должно получиться: тип
и форма в аннотации, размеры входов и выходов в комментарии перед функцией.
Как считать — в теории перед задачей. Имена переменных и функций менять
нельзя, проверка ищет их по именам. После каждой задачи идет ячейка
с открытыми проверками; при сдаче работа дополнительно проверяется закрытыми
тестами на других данных, а эксперименты пересчитываются независимо.

**Данные.** Основной датасет — перепись населения США 1994 года, Adult:
почти пятьдесят тысяч взрослых, возраст, образование, работа, семейное
положение, и ответ — зарабатывает ли человек больше 50 тысяч долларов в год.
На таких данных банки оценивают платежеспособность, а исследователи проверяют,
не ведет ли себя модель по-разному с разными группами людей. В последнем
блоке — сообщения из новостных групп про компьютеры, электронику, медицину
и объявления о продаже.

При первом запуске нужен интернет: данные и словари `nltk` скачиваются
и дальше лежат в кэше.

In [ ]:
import os
import re
import warnings

import joblib
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from scipy.stats import rankdata
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_20newsgroups, fetch_openml
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV, GroupKFold, StratifiedKFold, cross_val_score, train_test_split,
)
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, TargetEncoder

warnings.filterwarnings("ignore", category=ConvergenceWarning)
nltk.download("stopwords", quiet=True)

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

BLUE, BLACK, OCHRE, GREY = "#0072CE", "#0F1418", "#C98A3C", "#9CA3AF"
SEED = 42

In [ ]:
raw = fetch_openml(data_id=1590, as_frame=True, parser="auto").frame
income = (raw["class"] == ">50K").astype(int).to_numpy()   # 1 — больше 50 тысяч
features_raw = raw.drop(columns="class")

raw_train, raw_test, y_train, y_test = train_test_split(
    features_raw, income, test_size=0.25, random_state=SEED, stratify=income
)
print("строк:", raw.shape[0], " train:", len(raw_train), " test:", len(raw_test))
print(f"доля зарабатывающих больше 50 тысяч: {income.mean():.3f}")
raw.head()

## Блок A. Первый взгляд на данные

### Задача 1. Какие пропуски встречаются вместе

Пропуски редко бывают независимыми. Если два столбца пропущены в одних
и тех же строках, скорее всего, у пропуска одна причина, и ее стоит найти
до того, как чем-то заполнять.

Матрица совместных пропусков: строки и столбцы — столбцы таблицы, в которых
есть хотя бы один пропуск, в том же порядке, что в таблице. На пересечении
строки $a$ и столбца $b$ — число строк, где пропущены и $a$, и $b$.
На диагонали получается число пропусков в самом столбце.

In [ ]:
# df: (n, m) — любая таблица
# возвращает (k, k), где k — число столбцов с пропусками;
# индекс и столбцы — имена этих столбцов в порядке df, значения — целые
def missing_cooccurrence(df: pd.DataFrame) -> pd.DataFrame:
    return ...

In [ ]:
# --- проверка ---
toy = pd.DataFrame({
    "a": [1.0, np.nan, np.nan, 4.0],
    "b": ["x", "y", "z", "w"],
    "c": [np.nan, np.nan, 3.0, 4.0],
})
toy_result = missing_cooccurrence(toy)
assert list(toy_result.index) == ["a", "c"] and list(toy_result.columns) == ["a", "c"], "в матрице только столбцы с пропусками, в порядке таблицы"
assert np.array_equal(toy_result.to_numpy(), [[2, 1], [1, 2]]), "на диагонали — пропуски в столбце, вне ее — пропуски в обоих сразу"

adult_missing = missing_cooccurrence(raw)
print(adult_missing)
assert list(adult_missing.index) == ["workclass", "occupation", "native-country"]
assert adult_missing.loc["workclass", "occupation"] == 2799
print("проверки пройдены")

Работодатель и профессия почти всегда пропущены вместе, но профессия
пропущена на десять строк чаще. Посмотрим, кто эти десять человек.

In [ ]:
only_occupation = raw["occupation"].isna() & raw["workclass"].notna()
print(raw.loc[only_occupation, ["age", "workclass", "occupation", "hours-per-week", "class"]])

**Вопрос.** Все десять никогда не работали, поэтому профессии у них нет.
Что это говорит о пропусках в `workclass` и `occupation` вообще: это случайная
потеря данных или признак? Чем тогда лучше их заполнить: самым частым
значением или отдельной категорией?

*Ответ пишите здесь.*

### Задача 2, бонус. Столбцы, которые повторяют друг друга

Два столбца несут одну и ту же информацию, если их значения взаимно
однозначно соответствуют друг другу: каждому значению первого отвечает
ровно одно значение второго и наоборот. Проверяется это через число
различных значений: у столбца $a$, у столбца $b$ и у пар $(a, b)$ оно
должно совпадать. Пропуск считается отдельным значением.

Функция возвращает все такие пары столбцов. В каждой паре первым идет
столбец, который в таблице левее; пары идут в порядке первого столбца,
а при равенстве — второго.

In [ ]:
# df: (n, m)
# возвращает список пар имен столбцов (a, b), где a левее b в таблице
def one_to_one_pairs(df: pd.DataFrame) -> list[tuple[str, str]]:
    return ...

In [ ]:
# --- проверка ---
toy = pd.DataFrame({
    "code": [1, 2, 2, 3],
    "name": ["one", "two", "two", "three"],
    "group": ["a", "a", "b", "b"],
    "flag": [np.nan, 0.0, 1.0, 1.0],
})
assert one_to_one_pairs(toy) == [("code", "name")], "code и name соответствуют друг другу, остальные пары — нет"
toy_nan = pd.DataFrame({"x": [np.nan, 1.0, 1.0], "y": ["n", "o", "o"]})
assert one_to_one_pairs(toy_nan) == [("x", "y")], "пропуск — отдельное значение"

adult_pairs = one_to_one_pairs(raw)
print(adult_pairs)
assert adult_pairs == [("education", "education-num")]
print("проверки пройдены")

In [ ]:
print(raw.groupby("education", observed=True)["education-num"].first().sort_values())

`education-num` — это просто номер ступени образования, от дошкольного
до докторской степени. Один из двух столбцов лишний, и удобнее оставить
числовой: порядок ступеней в нем уже записан.

### Задача 3. Чистка

Функция готовит таблицу к анализу. Исходную таблицу она не меняет,
а возвращает новую, в которой:

- удалены столбцы `fnlwgt`, `education` и `class`, если они есть.
  `fnlwgt` — вес наблюдения в переписи: сколько похожих людей представляет
  эта строка. Это свойство выборки, а не человека;
- в каждом нечисловом столбце пропуски заменены строкой `"Unknown"`,
  а сам столбец приведен к обычным строкам, без типа `category`;
- числовые столбцы не тронуты.

In [ ]:
# df: (n, m) — строки в формате raw, столбец class может быть, а может не быть
# возвращает новую таблицу (n, m'): те же строки в том же порядке
def clean_adult(df: pd.DataFrame) -> pd.DataFrame:
    return ...

In [ ]:
# --- проверка ---
before = raw_train.head(50).copy()
cleaned_check = clean_adult(raw_train.head(50))
assert raw_train.head(50).equals(before), "исходная таблица не должна меняться"
assert "fnlwgt" not in cleaned_check and "education" not in cleaned_check, "fnlwgt и education удаляются"
assert "education-num" in cleaned_check and "age" in cleaned_check
assert "class" not in clean_adult(raw.head(5)), "class удаляется, если он есть"
assert cleaned_check.index.equals(raw_train.head(50).index), "строки и их порядок не меняются"
assert not isinstance(cleaned_check["workclass"].dtype, pd.CategoricalDtype), "тип category заменяется обычными строками"
full_clean = clean_adult(raw_train)
assert full_clean.isna().sum().sum() == 0, "после чистки пропусков не остается"
assert (full_clean["occupation"] == "Unknown").sum() == raw_train["occupation"].isna().sum()
assert pd.api.types.is_integer_dtype(full_clean["age"]), "числовые столбцы не трогаем"
print(full_clean.shape)
print("проверки пройдены")

Дальше в анализе используем очищенное обучение и два списка признаков.

In [ ]:
train_clean = clean_adult(raw_train)
numeric_columns = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
categorical_columns = ["workclass", "marital-status", "occupation", "relationship", "race", "sex", "native-country"]
assert list(train_clean.columns) == [c for c in raw_train.columns if c in numeric_columns + categorical_columns]

print(train_clean["native-country"].value_counts().tail(12))

### Задача 4. Редкие категории

В `native-country` сорок одна страна, и у многих в обучении меньше
пятидесяти человек. Вес при такой категории оценивается по горстке
объектов, а в новых данных вдобавок появятся страны, которых в обучении
не было вовсе. Обычное решение — собрать все редкие значения в одну
категорию.

Функция получает значения столбца в обучении и значения, которые нужно
преобразовать. Значение, которое в обучении встретилось меньше `min_count`
раз или не встретилось вовсе, заменяется на `other`. Остальные остаются
как есть. Возвращается `pd.Series` с тем же индексом, что у `values`.

In [ ]:
# train_values: (n_train,) — строки; values: (n,) — строки
# возвращает (n,) с индексом values
def group_rare(train_values: pd.Series, values: pd.Series, min_count: int, other: str = "Other") -> pd.Series:
    return ...

In [ ]:
# --- проверка ---
toy_train = pd.Series(["a", "a", "a", "b", "b", "c"])
toy_values = pd.Series(["a", "b", "c", "d"], index=[10, 11, 12, 13])
toy_grouped = group_rare(toy_train, toy_values, min_count=2)
assert toy_grouped.index.equals(toy_values.index), "индекс как у values"
assert list(toy_grouped) == ["a", "b", "Other", "Other"], "редкое c и незнакомое d становятся Other"
assert list(group_rare(toy_train, toy_values, 3, other="rare")) == ["a", "rare", "rare", "rare"]

test_clean = clean_adult(raw_test)
countries_grouped = group_rare(train_clean["native-country"], test_clean["native-country"], min_count=100)
print(countries_grouped.value_counts())
train_counts = train_clean["native-country"].value_counts()
kept_countries = set(countries_grouped) - {"Other"}
assert all(train_counts[country] >= 100 for country in kept_countries), "остаются только страны, у которых в обучении не меньше 100 человек"
assert countries_grouped.nunique() == (train_counts >= 100).sum() + 1
print("проверки пройдены")

Внутри пайплайна то же самое делает `OneHotEncoder(min_frequency=100,
handle_unknown="infrequent_if_exist")`: редкие категории он складывает
в один столбец, а незнакомые в тесте отправляет туда же.

## Блок B. Связь признаков с ответом

### Задача 5. Сила связи двух категориальных признаков

Для двух числовых признаков есть корреляция. Для двух категориальных нужна
другая мера. Возьмем таблицу сопряженности: $n_{ij}$ — число объектов,
у которых первый признак равен $i$-му значению, а второй — $j$-му. Суммы
по строкам и столбцам обозначим $n_{i\cdot} = \sum_j n_{ij}$
и $n_{\cdot j} = \sum_i n_{ij}$, всего объектов $n$.

**Сколько было бы при независимости.** Если признаки независимы, то
$P(a = i,\ b = j) = P(a = i)\,P(b = j)$. Оценим вероятности долями:
$P(a = i) \approx n_{i\cdot} / n$ и $P(b = j) \approx n_{\cdot j} / n$.
Тогда ожидаемое число объектов в клетке

$$E_{ij} = n \cdot \frac{n_{i\cdot}}{n} \cdot \frac{n_{\cdot j}}{n} = \frac{n_{i\cdot}\, n_{\cdot j}}{n}.$$

**Насколько данные отличаются от независимости.** Статистика хи-квадрат
складывает квадраты отклонений, нормированные на ожидание:

$$\chi^2 = \sum_{i,j} \frac{(n_{ij} - E_{ij})^2}{E_{ij}}.$$

**Нормировка.** $\chi^2$ растет вместе с $n$ и с размером таблицы, поэтому
сравнивать его между парами признаков нельзя. Посчитаем, чему он равен
при самой сильной связи. Пусть у обоих признаков $k$ значений и первый
однозначно определяет второй: все объекты лежат на диагонали,
$n_{ii} = n_{i\cdot} = n_{\cdot i}$. Обозначим $p_i = n_{i\cdot}/n$, тогда
$E_{ij} = n\,p_i\,p_j$.

Диагональная клетка дает
$\dfrac{(n p_i - n p_i^2)^2}{n p_i^2} = n (1 - p_i)^2$, клетка вне диагонали,
где $n_{ij} = 0$, дает $\dfrac{(0 - n p_i p_j)^2}{n p_i p_j} = n p_i p_j$. Складываем:

$$\chi^2 = n \sum_i (1 - p_i)^2 + n \sum_{i \ne j} p_i p_j.$$

Раскроем первую сумму: $\sum_i (1 - 2p_i + p_i^2) = k - 2 + \sum_i p_i^2$,
потому что $\sum_i p_i = 1$. Во второй сумме
$\sum_{i \ne j} p_i p_j = \left(\sum_i p_i\right)^2 - \sum_i p_i^2 = 1 - \sum_i p_i^2$.
Вместе

$$\chi^2 = n\left(k - 2 + \sum_i p_i^2 + 1 - \sum_i p_i^2\right) = n\,(k - 1).$$

Если у признаков разное число значений, $r$ и $c$, максимум равен
$n(\min(r, c) - 1)$. Делим на него и извлекаем корень, чтобы мера была
в тех же единицах, что корреляция. Это **V Крамера**:

$$V = \sqrt{\frac{\chi^2}{n\,(\min(r, c) - 1)}}, \qquad 0 \le V \le 1.$$

$V = 0$ — признаки независимы в выборке, $V = 1$ — один однозначно
определяет другой. Если у одного из признаков всего одно значение,
связи нет и функция возвращает 0. Поправку Йейтса для таблиц 2 на 2
не применяем.

In [ ]:
# x, y: (n,) — значения двух признаков для одних и тех же объектов, в одном порядке
# возвращает число от 0 до 1
def cramers_v(x: pd.Series, y: pd.Series) -> float:
    return ...

In [ ]:
# --- проверка ---
from scipy.stats import chi2_contingency

a_check = pd.Series(["u", "u", "v", "v", "w", "w", "u", "v"])
b_check = pd.Series(["p", "q", "p", "p", "q", "q", "p", "q"])
table_check = pd.crosstab(a_check, b_check).to_numpy()
chi2_check = chi2_contingency(table_check, correction=False)[0]
assert np.isclose(cramers_v(a_check, b_check), np.sqrt(chi2_check / (8 * 1))), "V не совпал с формулой через chi2 из scipy"
assert np.isclose(cramers_v(a_check, a_check), 1.0), "признак сам с собой связан полностью"
assert cramers_v(a_check, pd.Series(["z"] * 8)) == 0.0, "с константой связи нет"
assert np.isclose(cramers_v(pd.Series([0, 0, 1, 1]), pd.Series([0, 1, 0, 1])), 0.0), "независимые в выборке признаки дают 0"
print("проверки пройдены")

### Задача 6, бонус. Какие признаки связаны с доходом и друг с другом

На `train_clean` посчитайте:

- `target_association` — V Крамера каждого признака из `categorical_columns`
  с ответом `y_train`: `pd.Series` с индексом-именем признака,
  по убыванию значения;
- `feature_association` — матрица V Крамера между признаками
  из `categorical_columns`: `pd.DataFrame`, строки и столбцы в порядке
  списка, на диагонали единицы;
- `most_redundant_pair` — пара разных признаков с самым большим V,
  в порядке, в котором они стоят в `categorical_columns`.

In [ ]:
target_association: pd.Series = ...       # индекс — признаки, по убыванию V
feature_association: pd.DataFrame = ...   # (7, 7), порядок categorical_columns
most_redundant_pair: tuple[str, str] = ...

print(target_association.round(3))
print("сильнее всего связаны:", most_redundant_pair)

In [ ]:
# --- проверка ---
assert set(target_association.index) == set(categorical_columns)
assert target_association.is_monotonic_decreasing, "target_association идет по убыванию"
assert np.isclose(target_association["sex"], cramers_v(train_clean["sex"], y_train))
assert feature_association.shape == (7, 7) and list(feature_association.index) == categorical_columns
assert np.allclose(np.diag(feature_association.to_numpy()), 1.0) and np.allclose(feature_association, feature_association.T)
a_pair, b_pair = most_redundant_pair
assert categorical_columns.index(a_pair) < categorical_columns.index(b_pair), "пара в порядке categorical_columns"
off_diagonal = feature_association.to_numpy()[~np.eye(7, dtype=bool)]
assert np.isclose(feature_association.loc[a_pair, b_pair], off_diagonal.max())
print("проверки пройдены")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
image = ax.imshow(feature_association.to_numpy(), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(7), categorical_columns, rotation=40, ha="right")
ax.set_yticks(range(7), categorical_columns)
for i in range(7):
    for j in range(7):
        value = feature_association.iloc[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", color="white" if value > 0.6 else BLACK, fontsize=8)
ax.grid(False)
plt.colorbar(image, ax=ax, label="V Крамера")
plt.title("Связь категориальных признаков между собой")
plt.tight_layout()
plt.show()
print(pd.crosstab(train_clean["relationship"], train_clean["sex"]))

**Вопрос.** Почему `relationship` так сильно связан с полом? Посмотрите
на таблицу сопряженности выше. Что это значит, когда мы смотрим на вес
признака `sex` в линейной модели: можно ли по нему одному судить,
как пол влияет на прогноз?

*Ответ пишите здесь.*

## Блок C. Кодирование категорий

### Задача 7, бонус. One-hot руками

Категории берутся из обучения: все различные значения `train_values`,
отсортированные по возрастанию. Каждому значению из `values` отвечает
строка из нулей с единицей в столбце его категории. Значение, которого
в обучении не было, дает строку из одних нулей. Пропусков во входах нет.

In [ ]:
# train_values: (n_train,), values: (n,) — строки без пропусков
# возвращает (n, число различных значений в train_values) из нулей и единиц
def one_hot(train_values: pd.Series, values: pd.Series) -> np.ndarray:
    return ...

In [ ]:
# --- проверка ---
toy_encoded = one_hot(pd.Series(["b", "a", "c", "a"]), pd.Series(["a", "c", "z"]))
assert toy_encoded.shape == (3, 3), "столбцов столько, сколько категорий в обучении"
assert np.array_equal(toy_encoded, [[1, 0, 0], [0, 0, 1], [0, 0, 0]]), "категории по алфавиту, незнакомое значение — нули"

reference = OneHotEncoder(handle_unknown="ignore").fit(train_clean[["occupation"]])
assert np.array_equal(
    one_hot(train_clean["occupation"], test_clean["occupation"]),
    reference.transform(test_clean[["occupation"]]).toarray(),
), "не совпало с OneHotEncoder из sklearn"
print("проверки пройдены")

### Задача 8. Три способа закодировать категории

Сравним кодировки в одном сетапе: одна модель, одни данные, одна метрика.
Напишите функцию `make_adult_model(encoding)`, которая собирает пайплайн

```
make_pipeline(
    ColumnTransformer([
        ("num", StandardScaler(), numeric_columns),
        ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="Unknown"), <кодировщик>), categorical_columns),
    ]),
    LogisticRegression(max_iter=2000),
)
```

Кодировщик выбирается по строке `encoding` и создается заново при каждом
вызове:

| `encoding` | кодировщик |
|:--|:--|
| `"onehot"` | `OneHotEncoder(handle_unknown="ignore")` |
| `"ordinal"` | `OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)` |
| `"target"` | `TargetEncoder(cv=StratifiedKFold(5, shuffle=True, random_state=SEED))` |

Пайплайн принимает таблицу в сыром виде, как `raw_train`: лишние столбцы
`fnlwgt` и `education` отбрасываются сами, потому что их нет в списках,
а пропуски заполняются внутри. Чистка оказывается частью модели, и это
пригодится в блоке F.

Для каждой кодировки посчитайте средний ROC-AUC на кросс-валидации
`cv_adult` по `raw_train`, `y_train` и сложите в словарь `encoding_scores`.
В `encoding_n_features` положите, сколько признаков получает логистическая
регрессия: число столбцов после первого шага пайплайна, обученного
на всем `raw_train`.

In [ ]:
cv_adult = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
ENCODINGS = ["onehot", "ordinal", "target"]

In [ ]:
# encoding: "onehot", "ordinal" или "target"
# возвращает необученный Pipeline из make_pipeline: ColumnTransformer, затем LogisticRegression
def make_adult_model(encoding: str) -> Pipeline:
    return ...


...  # ваш код: кросс-валидация и число признаков для каждой кодировки

encoding_scores: dict[str, float] = ...     # ключи ENCODINGS, значения — средний ROC-AUC
encoding_n_features: dict[str, int] = ...   # ключи ENCODINGS

for encoding in ENCODINGS:
    print(f"{encoding:8s} ROC-AUC {encoding_scores[encoding]:.4f}, признаков {encoding_n_features[encoding]}")

In [ ]:
# --- проверка ---
model_check = make_adult_model("onehot")
assert isinstance(model_check, Pipeline) and isinstance(model_check[-1], LogisticRegression), "пайплайн заканчивается логистической регрессией"
assert not hasattr(model_check[-1], "coef_"), "функция возвращает необученный пайплайн"
assert make_adult_model("target") is not make_adult_model("target"), "каждый вызов собирает новый пайплайн"
assert set(encoding_scores) == set(ENCODINGS) and set(encoding_n_features) == set(ENCODINGS)
assert encoding_n_features["ordinal"] == encoding_n_features["target"] == len(numeric_columns) + len(categorical_columns)
assert encoding_n_features["onehot"] > 80
assert encoding_scores["onehot"] > encoding_scores["ordinal"], "для линейной модели one-hot должен выиграть у порядковой кодировки"
print("проверки пройдены")

**Вопрос.** Порядковая кодировка проигрывает больше пяти сотых ROC-AUC.
Почему линейной модели она не подходит: что модель вынуждена предполагать
про профессии, если профессия записана номером? Когда целевое кодирование
выгоднее one-hot, хотя здесь они почти равны?

*Ответ пишите здесь.*

## Блок D. Метрики

### Задача 9, бонус. ROC-AUC через ранги

ROC-AUC — это вероятность того, что случайный объект положительного класса
получит оценку выше, чем случайный объект отрицательного; при равных
оценках пара засчитывается наполовину. Если $n_1$ положительных и $n_0$
отрицательных, пар всего $n_1 n_0$, и

$$\text{AUC} = \frac{1}{n_1 n_0} \sum_{i:\, y_i = 1} \Big( \#\{j:\, y_j = 0,\ s_j < s_i\} + \tfrac{1}{2}\,\#\{j:\, y_j = 0,\ s_j = s_i\} \Big).$$

Перебор пар стоит $O(n_1 n_0)$. Через ранги то же самое считается
за сортировку.

**Ранги.** Отсортируем все $n$ оценок по возрастанию и пронумеруем
с единицы. Одинаковым оценкам дадим средний номер: это делает
`scipy.stats.rankdata`. Пусть у объекта $i$ оценка $s_i$, $L_i$ объектов
имеют оценку строго меньше, а $T_i$ объектов, включая сам $i$, — ровно
такую же. Они занимают номера с $L_i + 1$ по $L_i + T_i$, среднее
этих номеров

$$r_i = L_i + \frac{T_i + 1}{2}.$$

**Что считает сумма рангов.** Вычтем единицу:
$r_i - 1 = L_i + \tfrac{1}{2}(T_i - 1)$. Это ровно число объектов ниже $i$
плюс половина равных ему, не считая его самого, причем объектов обоих
классов. Сложим по всем положительным:

$$\sum_{i:\, y_i = 1} (r_i - 1) = \underbrace{\text{вклад пар «положительный — отрицательный»}}_{\text{то, что нужно}} + \text{вклад пар «положительный — положительный»}.$$

**Лишнее слагаемое.** Возьмем два положительных объекта. Если их оценки
разные, пара дает единицу тому, кто выше. Если равные, каждый получает
по половине. В любом случае пара вносит ровно 1, а таких пар
$n_1 (n_1 - 1) / 2$. Значит,

$$\text{AUC} = \frac{1}{n_1 n_0}\left(\sum_{i:\, y_i = 1} r_i - n_1 - \frac{n_1 (n_1 - 1)}{2}\right) = \frac{1}{n_1 n_0}\left(\sum_{i:\, y_i = 1} r_i - \frac{n_1 (n_1 + 1)}{2}\right).$$

In [ ]:
# y_true: (n,) из нулей и единиц, оба класса есть; scores: (n,) — чем больше, тем увереннее в классе 1
# возвращает число от 0 до 1
def roc_auc_rank(y_true: np.ndarray, scores: np.ndarray) -> float:
    return ...

In [ ]:
# --- проверка ---
# 1. счет руками: пары (0.8 > 0.3), (0.8 > 0.5), (0.5 > 0.3), (0.5 = 0.5) пополам — 3.5 из 4
assert np.isclose(roc_auc_rank(np.array([1, 0, 1, 0]), np.array([0.8, 0.3, 0.5, 0.5])), 3.5 / 4), "пример со связкой посчитан неверно"
# 2. совпадение с sklearn, в том числе при большом числе равных оценок
rng_check = np.random.default_rng(0)
y_rand = rng_check.integers(0, 2, 500)
s_rand = np.round(rng_check.normal(size=500) + y_rand, 1)
assert np.isclose(roc_auc_rank(y_rand, s_rand), roc_auc_score(y_rand, s_rand)), "не совпало с roc_auc_score"
# 3. крайние случаи
assert np.isclose(roc_auc_rank(np.array([0, 0, 1, 1]), np.array([1, 2, 3, 4])), 1.0)
assert np.isclose(roc_auc_rank(np.array([0, 1, 0, 1]), np.array([5, 5, 5, 5])), 0.5), "все оценки равны — половина"
print("проверки пройдены")

### Задача 10. Метрики по группам

Одно число на всем тесте прячет, где модель ошибается. В домашней работе
понадобится таблица метрик по темам, здесь — по группам людей.

Функция получает ответы, вероятности класса 1 и группу каждого объекта.
Метки получаются порогом: класс 1, если вероятность не меньше `threshold`.
Для каждой группы считается строка таблицы:

| столбец | что это |
|:--|:--|
| `size` | число объектов в группе |
| `positive_share` | доля класса 1 среди них |
| `precision`, `recall`, `f1` | по меткам; если делить не на что, 0 |
| `roc_auc` | по вероятностям; если в группе один класс, `NaN` |

Индекс таблицы — значения группы по возрастанию.

In [ ]:
# y_true: (n,) из нулей и единиц, proba: (n,) — вероятности класса 1, groups: (n,) — группа объекта
# возвращает (число групп, 6): столбцы size, positive_share, precision, recall, f1, roc_auc
def metrics_by_group(y_true: np.ndarray, proba: np.ndarray, groups: pd.Series, threshold: float = 0.5) -> pd.DataFrame:
    return ...

In [ ]:
# --- проверка ---
y_toy = np.array([1, 0, 1, 1, 0, 0, 1])
p_toy = np.array([0.9, 0.6, 0.4, 0.7, 0.2, 0.1, 0.8])
g_toy = pd.Series(["b", "b", "b", "a", "a", "c", "c"])
table_toy = metrics_by_group(y_toy, p_toy, g_toy)
assert list(table_toy.index) == ["a", "b", "c"], "группы по возрастанию"
assert list(table_toy.columns) == ["size", "positive_share", "precision", "recall", "f1", "roc_auc"]
assert list(table_toy["size"]) == [2, 3, 2]
# группа b: ответы 1, 0, 1; метки 1, 1, 0 — precision 1/2, recall 1/2
assert np.isclose(table_toy.loc["b", "precision"], 0.5) and np.isclose(table_toy.loc["b", "recall"], 0.5)
assert np.isclose(table_toy.loc["b", "roc_auc"], 0.5), "в группе b одна пара из двух упорядочена верно"
assert table_toy.loc["a", "roc_auc"] == 1.0
assert np.isclose(metrics_by_group(y_toy, p_toy, g_toy, threshold=0.75).loc["a", "recall"], 0.0), "порог — параметр"
single = metrics_by_group(np.array([1, 1]), np.array([0.3, 0.9]), pd.Series(["x", "x"]))
assert np.isnan(single.loc["x", "roc_auc"]), "при одном классе ROC-AUC не определен"
print("проверки пройдены")

### Задача 11. Базовые модели

Любую модель надо сравнивать с тем, что получается без нее. Обучите
на `raw_train` четыре модели и посчитайте их метрики на `raw_test`:

| имя | модель |
|:--|:--|
| `most_frequent` | `DummyClassifier(strategy="most_frequent")` — всегда самый частый класс |
| `stratified` | `DummyClassifier(strategy="stratified", random_state=SEED)` — случайные ответы в пропорции классов |
| `numeric_only` | `make_pipeline(ColumnTransformer([("num", StandardScaler(), numeric_columns)]), LogisticRegression(max_iter=2000))` |
| `onehot_logreg` | `make_adult_model("onehot")` |

Обученные модели сложите в словарь `baseline_models`, метрики —
в таблицу `baseline_table`: строки в порядке таблицы выше, столбцы
`accuracy`, `balanced_accuracy`, `f1` по меткам `predict`, `roc_auc`
и `pr_auc` по вероятности класса 1 из `predict_proba`. `pr_auc` считайте
как `average_precision_score`.

In [ ]:
BASELINES = ["most_frequent", "stratified", "numeric_only", "onehot_logreg"]
METRICS = ["accuracy", "balanced_accuracy", "f1", "roc_auc", "pr_auc"]

In [ ]:
baseline_models: dict = ...          # ключи BASELINES, значения — обученные модели

...  # ваш код: метрики каждой модели на тесте

baseline_table: pd.DataFrame = ...   # (4, 5): строки BASELINES, столбцы METRICS
print(baseline_table.round(3))

In [ ]:
# --- проверка ---
assert list(baseline_table.index) == BASELINES and list(baseline_table.columns) == METRICS
assert all(hasattr(baseline_models[name], "predict_proba") for name in BASELINES)
assert np.isclose(baseline_table.loc["most_frequent", "accuracy"], 1 - y_test.mean()), "самый частый класс угадывает долю нулей"
assert np.isclose(baseline_table.loc["most_frequent", "roc_auc"], 0.5) and baseline_table.loc["most_frequent", "f1"] == 0
assert np.isclose(baseline_table.loc["most_frequent", "pr_auc"], y_test.mean()), "PR-AUC константы равен доле класса 1"
assert abs(baseline_table.loc["stratified", "roc_auc"] - 0.5) < 0.03
assert baseline_table.loc["onehot_logreg", "roc_auc"] > baseline_table.loc["numeric_only", "roc_auc"] > 0.75
print("проверки пройдены")

Accuracy у константы 0.76: число выглядит прилично, пока рядом нет второй
строки. Сбалансированная accuracy, F1 и PR-AUC сразу показывают, что
константа ничего не умеет. PR-AUC у нее равен доле класса 1, и это
нижняя граница, с которой надо сравнивать PR-AUC любой модели.

### Задача 12. Одинаково ли модель работает для разных людей

Посчитайте `metrics_by_group` для модели `onehot_logreg` на тесте
по полу — `sex_table` — и по расе — `race_table`. В `recall_gap_sex`
положите разницу между наибольшим и наименьшим recall в `sex_table`.

In [ ]:
test_proba: np.ndarray = ...     # (n_test,) — вероятности класса 1 от onehot_logreg
sex_table: pd.DataFrame = ...
race_table: pd.DataFrame = ...
recall_gap_sex: float = ...

print(sex_table.round(3))
print(race_table.round(3))
print(f"разрыв recall по полу: {recall_gap_sex:.3f}")

In [ ]:
# --- проверка ---
assert test_proba.shape == (len(y_test),)
assert list(sex_table.index) == ["Female", "Male"]
assert sex_table["size"].sum() == len(y_test) and race_table["size"].sum() == len(y_test)
assert np.isclose(sex_table.loc["Female", "recall"], recall_score(y_test[raw_test["sex"].to_numpy() == "Female"], (test_proba >= 0.5).astype(int)[raw_test["sex"].to_numpy() == "Female"]))
assert np.isclose(recall_gap_sex, abs(sex_table.loc["Male", "recall"] - sex_table.loc["Female", "recall"]))
print("проверки пройдены")

**Вопрос.** У женщин recall заметно ниже, а ROC-AUC при этом не хуже.
Как такое возможно: что значит высокий ROC-AUC при низком recall? Посмотрите
на `positive_share` в обеих группах. Что поменялось бы, если бы порог
выбирался для каждой группы отдельно, и почему такое решение само по себе
спорное?

*Ответ пишите здесь.*

## Блок E. Валидация

### Задача 13, бонус. Стратифицированное разбиение руками

Разбиение на `n_splits` частей, в каждой из которых доля классов почти
такая же, как во всей выборке. Алгоритм:

1. `rng = np.random.default_rng(seed)`;
2. для каждого класса по возрастанию его метки берем индексы объектов
   этого класса и перемешиваем их `rng.permutation`, по одному вызову
   на класс в этом порядке;
3. выписываем перемешанные индексы всех классов подряд, класс за классом,
   и раздаем по кругу: объект на месте $j$ этого списка попадает
   в часть $j \bmod$ `n_splits`, считая места с нуля.

Раздача продолжается с того места, где закончился предыдущий класс,
поэтому размеры частей отличаются не больше чем на единицу.

Функция возвращает список из `n_splits` пар `(train_idx, test_idx)`,
где `test_idx` — индексы части $k$, а `train_idx` — все остальные.
Оба массива отсортированы по возрастанию. Такой список можно прямо
передать в `cross_val_score(..., cv=...)`.

In [ ]:
# y: (n,) — метки классов; возвращает список длины n_splits,
# в каждой паре train_idx: (n - n_k,), test_idx: (n_k,) — отсортированные целые индексы
def stratified_folds(y: np.ndarray, n_splits: int, seed: int) -> list[tuple[np.ndarray, np.ndarray]]:
    return ...

In [ ]:
# --- проверка ---
y_toy = np.array([0, 1, 0, 0, 1, 0, 2, 0, 1, 0, 2, 0])
folds_toy = stratified_folds(y_toy, 3, seed=0)
assert len(folds_toy) == 3
tests = np.concatenate([test for _, test in folds_toy])
assert np.array_equal(np.sort(tests), np.arange(len(y_toy))), "каждый объект ровно один раз попадает в проверку"
for train, test in folds_toy:
    assert np.array_equal(np.sort(np.r_[train, test]), np.arange(len(y_toy))) and len(np.intersect1d(train, test)) == 0
    assert np.all(np.diff(test) > 0) and np.all(np.diff(train) > 0), "индексы отсортированы"
sizes = [len(test) for _, test in folds_toy]
assert max(sizes) - min(sizes) <= 1, "размеры частей отличаются не больше чем на единицу"

# первый класс раздается первым: его перемешанные индексы лягут в части 0, 1, 2, 0, ...
first_class = np.random.default_rng(0).permutation(np.flatnonzero(y_toy == 0))
assert first_class[0] in folds_toy[0][1] and first_class[1] in folds_toy[1][1], "порядок раздачи не совпал с описанием"

folds_adult = stratified_folds(y_train, 5, SEED)
shares = [y_train[test].mean() for _, test in folds_adult]
assert max(shares) - min(shares) < 0.001, "доля класса 1 в частях почти одинакова"
print("проверки пройдены")

### Задача 14, бонус. Разбиение по группам

Если у объектов есть группы — один пользователь, один пациент, один
и тот же запрос с разными ответами, — группа целиком должна попасть
либо в обучение, либо в проверку. Алгоритм:

1. `rng = np.random.default_rng(seed)`;
2. различные значения `groups` по возрастанию перемешиваем
   одним вызовом `rng.permutation`;
3. группа на месте $j$ перемешанного списка попадает в часть
   $j \bmod$ `n_splits`, а с ней все ее объекты.

Формат результата тот же, что в задаче 13.

In [ ]:
# groups: (n,) — группа каждого объекта; формат результата как у stratified_folds
def group_folds(groups: np.ndarray, n_splits: int, seed: int) -> list[tuple[np.ndarray, np.ndarray]]:
    return ...

In [ ]:
# --- проверка ---
g_toy = np.array(["u1", "u2", "u1", "u3", "u4", "u2", "u5", "u3", "u6"])
folds_toy = group_folds(g_toy, 3, seed=1)
assert len(folds_toy) == 3
assert np.array_equal(np.sort(np.concatenate([test for _, test in folds_toy])), np.arange(len(g_toy)))
for train, test in folds_toy:
    assert not set(g_toy[train]) & set(g_toy[test]), "группа не должна оказаться сразу в обучении и в проверке"
    assert np.all(np.diff(test) > 0) and np.all(np.diff(train) > 0)
shuffled_toy = np.random.default_rng(1).permutation(np.unique(g_toy))
assert set(g_toy[folds_toy[0][1]]) == {shuffled_toy[0], shuffled_toy[3]}, "в часть 0 попадают группы на местах 0 и 3"
print("проверки пройдены")

### Задача 15. Утечка через близнецов

Смоделируем ситуацию из домашней работы: у каждого объекта в выборке
есть почти точная копия. В логах так выглядят один и тот же запрос,
заданный несколько раз, или несколько похожих диалогов одного
пользователя. Возьмем две тысячи человек из обучения, числовые признаки
после стандартизации, и к каждому добавим копию с небольшим шумом.
Копия получает тот же номер группы, что оригинал.

Модель — один ближайший сосед, `KNeighborsClassifier(n_neighbors=1)`:
она просто запоминает обучение, поэтому утечку покажет ярче всего.

Посчитайте:

- `leaky_accuracy` — среднюю accuracy на кросс-валидации по `twin_X`,
  `twin_y` со случайным разбиением `StratifiedKFold(5, shuffle=True, random_state=SEED)`;
- `honest_accuracy` — то же с `GroupKFold(5)` по группам `twin_groups`;
- `test_accuracy` — accuracy модели, обученной на всех `twin_X`, `twin_y`,
  на настоящих новых людях `check_X`, `y_test`.

In [ ]:
rng_twins = np.random.default_rng(SEED)
twin_scaler = StandardScaler().fit(raw_train[numeric_columns].to_numpy(dtype=float))
base_idx = rng_twins.choice(len(raw_train), size=2000, replace=False)
base_X = twin_scaler.transform(raw_train[numeric_columns].to_numpy(dtype=float)[base_idx])
base_y = y_train[base_idx]

twin_X = np.vstack([base_X, base_X + rng_twins.normal(0, 0.05, base_X.shape)])
twin_y = np.r_[base_y, base_y]
twin_groups = np.r_[np.arange(2000), np.arange(2000)]
check_X = twin_scaler.transform(raw_test[numeric_columns].to_numpy(dtype=float))
print(twin_X.shape, check_X.shape)

In [ ]:
leaky_accuracy: float = ...
honest_accuracy: float = ...
test_accuracy: float = ...

print(f"случайное разбиение {leaky_accuracy:.3f}, по группам {honest_accuracy:.3f}, новые люди {test_accuracy:.3f}")

In [ ]:
# --- проверка ---
assert leaky_accuracy > honest_accuracy + 0.05, "случайное разбиение должно заметно завышать оценку"
assert abs(honest_accuracy - test_accuracy) < abs(leaky_accuracy - test_accuracy), "разбиение по группам должно быть ближе к правде"
print("проверки пройдены")

**Вопрос.** Случайное разбиение обещает accuracy на девять сотых выше,
чем модель покажет на новых людях. В первой домашней работе где у вас
могут оказаться такие близнецы, и что тогда будет группой?

*Ответ пишите здесь.*

### Задача 16. Подбор параметра и честная оценка

Подберите `C` для `make_adult_model("onehot")` перебором по сетке
`C_grid` с `GridSearchCV`, разбиением `cv_adult` и метрикой `"roc_auc"`
на `raw_train`. Имя параметра — `"logisticregression__C"`: так
`make_pipeline` называет шаг с логистической регрессией.

Сохраните:

- `search` — обученный `GridSearchCV`;
- `cv_table` — `pd.Series` с индексом `C` и средним ROC-AUC на
  кросс-валидации;
- `best_C` и `cv_best_score` — лучший `C` и его ROC-AUC на кросс-валидации;
- `test_score` — ROC-AUC лучшей модели на `raw_test`. Это единственное
  место в лабораторной, где тест используется после выбора модели.

In [ ]:
C_grid = [0.01, 0.1, 1, 10]

In [ ]:
search: GridSearchCV = ...
cv_table: pd.Series = ...      # индекс C_grid, значения — средний ROC-AUC
best_C: float = ...
cv_best_score: float = ...
test_score: float = ...

print(cv_table.round(4))
print(f"лучший C = {best_C}: кросс-валидация {cv_best_score:.4f}, тест {test_score:.4f}")

In [ ]:
# --- проверка ---
assert hasattr(search, "best_estimator_"), "search должен быть обучен"
assert list(cv_table.index) == C_grid
assert best_C == cv_table.idxmax() and np.isclose(cv_best_score, cv_table.max())
assert abs(test_score - cv_best_score) < 0.01, "оценка на тесте должна быть близка к кросс-валидации"
print("проверки пройдены")

## Блок F. Сохранение модели

### Задача 17. Модель как файл

Обученная модель нужна не только в ноутбуке, где ее обучили. В домашней
работе проверяющий код загрузит вашу модель у себя и подаст ей свои данные.
Для этого файл должен содержать все, что модель делает с сырыми данными:
выбор столбцов, заполнение пропусков, кодирование, масштабирование.
Пайплайн из блока C так и устроен, поэтому сохраняется одним файлом.

Напишите три функции:

- `save_model(model, path)` — сохраняет модель через `joblib.dump`,
  создав папку, если ее нет;
- `load_model(path)` — загружает модель через `joblib.load`;
- `predict_income(model, raw)` — вероятности класса 1 для таблицы
  в формате `raw`: столбцы могут идти в другом порядке, может быть
  столбец `class`, могут быть пропуски и незнакомые категории.

Сохраните лучшую модель из задачи 16, `search.best_estimator_`,
в `MODEL_PATH`.

In [ ]:
MODEL_PATH = os.path.join("artifacts", "adult_model.joblib")

In [ ]:
# model — обученный пайплайн, path — путь к файлу
def save_model(model, path: str) -> None:
    ...


def load_model(path: str):
    return ...


# raw: (n, m) — строки в формате raw; возвращает (n,) — вероятности класса 1
def predict_income(model, raw: pd.DataFrame) -> np.ndarray:
    return ...


...  # ваш код: сохранить лучшую модель в MODEL_PATH

In [ ]:
# --- проверка ---
assert os.path.exists(MODEL_PATH), "модель не сохранена"
loaded = load_model(MODEL_PATH)
assert np.allclose(predict_income(loaded, raw_test), search.best_estimator_.predict_proba(raw_test)[:, 1]), "загруженная модель предсказывает не так, как сохраненная"

strange = raw.sample(6, random_state=1)[list(reversed(raw.columns))].copy()   # другой порядок и столбец class
strange.loc[strange.index[0], "occupation"] = np.nan
strange["native-country"] = strange["native-country"].cat.add_categories(["Atlantis"])
strange.loc[strange.index[1], "native-country"] = "Atlantis"
strange_proba = predict_income(loaded, strange)
assert strange_proba.shape == (6,) and np.all((0 <= strange_proba) & (strange_proba <= 1))
print(strange_proba.round(3))
print("проверки пройдены")

Такой файл загружается в любом другом процессе, где стоит sklearn той же
версии. Есть одна ловушка. Если в пайплайне есть ваша собственная функция
или класс, например `FunctionTransformer(clean_adult)`, в файл попадает
не код функции, а только ее имя. Загрузить такой файл можно только там,
где эта функция тоже определена под тем же именем. Поэтому в домашней
работе собственный код предобработки кладите в отдельный модуль `.py`
и импортируйте его и при обучении, и при загрузке.

## Блок G. Тексты

### Задача 18. Нормализация слов

На семинаре слова приводились к основе по шагам. Соберите это в одну
функцию, которая работает и с английским, и с русским текстом:

1. перевести текст в нижний регистр;
2. выделить слова регулярным выражением `[^\W\d_]+`: подряд идущие буквы
   любого алфавита, без цифр и подчеркиваний;
3. выбросить слова короче двух букв;
4. выбросить стоп-слова из `stopwords.words(language)`;
5. привести каждое слово к основе `SnowballStemmer(language)`.

`language` — `"english"` или `"russian"`. Функция будет вызываться
на тысячах текстов: стеммер и множество стоп-слов не стоит создавать
заново при каждом вызове.

In [ ]:
# text — строка, language — "english" или "russian"
# возвращает список основ слов в порядке появления в тексте
def normalize_tokens(text: str, language: str) -> list[str]:
    return ...

In [ ]:
# --- проверка ---
en = normalize_tokens("The 3 cats were running faster than dogs_2 in 2024!", "english")
assert en == ["cat", "run", "faster", "dog"], f"английский пример: {en}"
ru = normalize_tokens("Кошки не любят мокрую погоду, а собаки — любят!", "russian")
assert ru == ["кошк", "люб", "мокр", "погод", "собак", "люб"], f"русский пример: {ru}"
assert normalize_tokens("", "russian") == [] and normalize_tokens("я и ты", "russian") == []
print("проверки пройдены")

### Задача 19. Сравнение текстовых моделей

Данные — сообщения из пяти новостных групп. Две из них, про железо
IBM PC и Macintosh, по смыслу очень близки. Чистка та же, что на семинаре:
выбрасываем тексты короче пяти слов и точные повторы.

In [ ]:
TOPICS = ["comp.sys.ibm.pc.hardware", "comp.sys.mac.hardware", "misc.forsale", "sci.electronics", "sci.med"]
news = fetch_20newsgroups(subset="all", categories=TOPICS, remove=("headers", "footers", "quotes"), random_state=SEED)
texts = pd.DataFrame({"text": news.data, "topic": np.array(news.target_names)[news.target]})
texts = texts[texts["text"].str.split().str.len() >= 5].drop_duplicates("text").reset_index(drop=True)
texts_train, texts_test = train_test_split(texts, test_size=0.25, random_state=SEED, stratify=texts["topic"])
print(len(texts_train), len(texts_test))
print(texts["topic"].value_counts())

Обучите на `texts_train["text"]`, `texts_train["topic"]` пять моделей
и оцените их на `texts_test`:

| имя | модель |
|:--|:--|
| `dummy` | `make_pipeline(CountVectorizer(), DummyClassifier(strategy="most_frequent"))` |
| `bow_nb` | `make_pipeline(CountVectorizer(min_df=2), MultinomialNB())` |
| `tfidf_logreg` | `make_pipeline(TfidfVectorizer(min_df=2), LogisticRegression(max_iter=2000))` |
| `tfidf_logreg_C10` | то же с `LogisticRegression(C=10, max_iter=2000)` |
| `tfidf_sublinear_C10` | `make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True), LogisticRegression(C=10, max_iter=2000))` |

`sublinear_tf=True` заменяет число вхождений $\text{tf}$ на $1 + \ln \text{tf}$:
слово, которое встретилось в тексте десять раз, весит не в десять раз
больше, чем встреченное один раз, а примерно в 3.3 раза.

Обученные модели сложите в `text_models`, метрики на тесте — в таблицу
`text_table`: строки в порядке таблицы выше, столбцы `accuracy`,
`macro_f1` и `n_features` — размер словаря первого шага пайплайна.

In [ ]:
TEXT_MODELS = ["dummy", "bow_nb", "tfidf_logreg", "tfidf_logreg_C10", "tfidf_sublinear_C10"]

In [ ]:
text_models: dict = ...        # ключи TEXT_MODELS, значения — обученные пайплайны

...  # ваш код: метрики каждой модели на тесте

text_table: pd.DataFrame = ... # (5, 3): строки TEXT_MODELS, столбцы accuracy, macro_f1, n_features
print(text_table.round(3))

In [ ]:
# --- проверка ---
assert list(text_table.index) == TEXT_MODELS and list(text_table.columns) == ["accuracy", "macro_f1", "n_features"]
assert text_table.loc["dummy", "macro_f1"] < 0.1
assert text_table.loc["bow_nb", "n_features"] == text_table.loc["tfidf_logreg", "n_features"], "min_df=2 в обоих, словари одинаковые"
assert text_table.loc["dummy", "n_features"] > text_table.loc["bow_nb", "n_features"], "без min_df словарь больше"
assert text_table.loc["tfidf_logreg_C10", "macro_f1"] > text_table.loc["tfidf_logreg", "macro_f1"]
assert text_table["macro_f1"].max() > 0.8
print("проверки пройдены")

**Вопрос.** С параметрами по умолчанию наивный Байес на простых счетчиках
обходит логистическую регрессию на TF-IDF. Что меняется при `C=10`
и почему по умолчанию модель, скорее всего, недообучена? Какой вывод
отсюда про сравнение моделей с параметрами по умолчанию?

*Ответ пишите здесь.*

### Задача 20. Разбор ошибок

Возьмите модель с наибольшим `macro_f1` из `text_table`, ее имя положите
в `best_text_model`. Для нее посчитайте:

- `text_confusion` — матрицу ошибок на тесте как `pd.DataFrame`: строки —
  настоящая тема, столбцы — предсказанная, обе оси в порядке `TOPICS`;
- `most_confused_pair` — пару разных тем, которые модель путает чаще всего,
  считая ошибки в обе стороны; темы в паре в порядке `TOPICS`.

Отдельно для модели `tfidf_sublinear_C10` соберите `top_terms`: словарь,
где для каждой темы из `TOPICS` лежит список десяти слов с наибольшим
весом этой темы в логистической регрессии, от большего веса к меньшему.

In [ ]:
best_text_model: str = ...
text_confusion: pd.DataFrame = ...            # (5, 5), индекс и столбцы TOPICS
most_confused_pair: tuple[str, str] = ...
top_terms: dict[str, list[str]] = ...         # ключи TOPICS, значения — по 10 слов

print(best_text_model)
print(text_confusion)
print("чаще всего путаются:", most_confused_pair)
for topic, words in top_terms.items():
    print(f"{topic:26s}", ", ".join(words))

In [ ]:
# --- проверка ---
assert best_text_model == text_table["macro_f1"].idxmax()
assert list(text_confusion.index) == TOPICS and list(text_confusion.columns) == TOPICS
assert text_confusion.to_numpy().sum() == len(texts_test)
assert text_confusion.loc["sci.med", "sci.med"] == np.sum((texts_test["topic"].to_numpy() == "sci.med") & (text_models[best_text_model].predict(texts_test["text"]) == "sci.med"))
a_topic, b_topic = most_confused_pair
assert TOPICS.index(a_topic) < TOPICS.index(b_topic)
assert set(top_terms) == set(TOPICS) and all(len(words) == 10 for words in top_terms.values())
print("проверки пройдены")

In [ ]:
errors = texts_test.assign(predicted=text_models[best_text_model].predict(texts_test["text"]))
errors = errors[(errors["topic"] == most_confused_pair[0]) & (errors["predicted"] == most_confused_pair[1])]
for _, row in errors.head(3).iterrows():
    print(f"тема {row['topic']}, модель сказала {row['predicted']}")
    print(row["text"][:400].strip(), "\n" + "-" * 60)

**Вопрос.** Прочитайте несколько ошибок выше. Модель ошиблась, или текст
и правда можно отнести к обеим темам? Что из этого следует для домашней
работы, где темы придумываете вы сами?

*Ответ пишите здесь.*

## Итог

После лабораторной вы должны уметь объяснить:

- почему пропуски сначала изучают и только потом заполняют, и как совместные
  пропуски подсказывают причину
- как найти столбцы, которые дублируют друг друга, и чем измерить связь
  двух категориальных признаков
- почему для линейной модели порядковая кодировка хуже one-hot и что
  делать с редкими и незнакомыми категориями
- как ROC-AUC связан с рангами и почему метрики нужно смотреть по группам
- зачем сравнивать модель с константой и что показывает PR-AUC константы
- как повторяющиеся объекты портят кросс-валидацию и как это чинит
  разбиение по группам
- как сохранить модель так, чтобы ее можно было загрузить и запустить
  на сырых данных в другом месте